# Title

Description

## Imports

In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
import torch
import json
import torch
import importlib

from tqdm import tqdm

from transformers import AutoTokenizer, AutoModel

from algorithm import Algorithm

from pathlib import Path

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import tensorflow as tf

## 1. Load dataframes

In [30]:
ai_train = pd.read_csv("../Dataframes/df_ai/df_ai_train.csv")
ai_val = pd.read_csv("../Dataframes/df_ai/df_ai_val.csv")
ai_test = pd.read_csv("../Dataframes/df_ai/df_ai_test.csv")

plagiarism_train = pd.read_csv("../Dataframes/df_plagiarism/df_plagiarism_train.csv")
plagiarism_val = pd.read_csv("../Dataframes/df_plagiarism/df_plagiarism_val.csv")
plagiarism_test = pd.read_csv("../Dataframes/df_plagiarism/df_plagiarism_test.csv")

print("------AI shapes------")
print("Train: ", ai_train.shape)
print("Validation: ", ai_val.shape)
print("Test: ", ai_test.shape)
print("------Plagiarism shapes------")
print("Train: ", plagiarism_train.shape)
print("Validation: ", plagiarism_val.shape)
print("Test: ", plagiarism_test.shape)

ai_train.head()

------AI shapes------
Train:  (24000, 20)
Validation:  (3000, 20)
Test:  (3000, 20)
------Plagiarism shapes------
Train:  (24000, 56)
Validation:  (3000, 56)
Test:  (3000, 56)


,code,label,approx_tokens,comment_density,avg_line_length,line_length_variance,blank_line_ratio,num_classes,num_methods,num_if,num_for,num_while,num_switch,o_complexity,max_depth,total_nodes,num_literals,num_ids,unique_ids,id_diversity
0,private Set<Integer> perNodeRelease(final C th...,0,101,0.0,61.882353,1268.339100,0.055556,0.0,0.0,0.0,0.0,0.0,0.0,1.0,6.0,0.0,0.0,63.0,30.0,0.476190
1,@Override\r\n public AuthenticationStatus f...,0,23,0.0,34.777778,702.395062,0.100000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,19.0,15.0,0.789474
2,public void callWorkListenerWithError(WorkCont...,1,28,0.0,49.500000,2072.250000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,5.0,0.0,0.0,7.0,7.0,1.000000
3,public void setSubscription(Subscription s) {\...,1,14,0.0,33.250000,500.187500,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,6.0,5.0,0.833333
4,public boolean getDialogContentInset(int theme...,1,20,0.0,44.666667,1461.222222,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,21.0,16.0,0.761905


## 2. Separate attribites and labels

In [31]:
X_ai_train = ai_train.drop("label", axis=1)
y_ai_train = ai_train["label"]

X_ai_val = ai_val.drop("label", axis=1)
y_ai_val = ai_val["label"]

X_ai_test = ai_test.drop("label", axis=1)
y_ai_test = ai_test["label"]

X_plagiarism_train = plagiarism_train.drop("label", axis=1)
y_plagiarism_train = plagiarism_train["label"]

X_plagiarism_val = plagiarism_val.drop("label", axis=1)
y_plagiarism_val = plagiarism_val["label"]

X_plagiarism_test = plagiarism_test.drop("label", axis=1)
y_plagiarism_test = plagiarism_test["label"]


print("X_ai_train:", X_ai_train.shape)
print("y_ai_train:", y_ai_train.shape)
print("X_ai_val:", X_ai_val.shape)
print("y_ai_val:", y_ai_val.shape)
print("X_ai_test:", X_ai_test.shape)
print("y_ai_test:", y_ai_test.shape)

print("X_plagiarism_train:", X_plagiarism_train.shape)
print("y_plagiarism_train:", y_plagiarism_train.shape)
print("X_plagiarism_val:", X_plagiarism_val.shape)
print("y_plagiarism_val:", y_plagiarism_val.shape)
print("X_plagiarism_test:", X_plagiarism_test.shape)
print("y_plagiarism_test:", y_plagiarism_test.shape)

print("\nTraining class distribution:")
print(y_ai_train.value_counts().sort_index())
print("\nValidation class distribution:")
print(y_ai_val.value_counts().sort_index())
print("\nTesting class distribution:")
print(y_ai_test.value_counts().sort_index())

print("\nTraining class distribution:")
print(y_plagiarism_train.value_counts().sort_index())
print("\nValidation class distribution:")
print(y_plagiarism_val.value_counts().sort_index())
print("\nTesting class distribution:")
print(y_plagiarism_test.value_counts().sort_index())


X_ai_train: (24000, 19)
y_ai_train: (24000,)
X_ai_val: (3000, 19)
y_ai_val: (3000,)
X_ai_test: (3000, 19)
y_ai_test: (3000,)
X_plagiarism_train: (24000, 55)
y_plagiarism_train: (24000,)
X_plagiarism_val: (3000, 55)
y_plagiarism_val: (3000,)
X_plagiarism_test: (3000, 55)
y_plagiarism_test: (3000,)

Training class distribution:
label
0    12000
1    12000
Name: count, dtype: int64

Validation class distribution:
label
0    1500
1    1500
Name: count, dtype: int64

Testing class distribution:
label
0    1500
1    1500
Name: count, dtype: int64

Training class distribution:
label
0    12000
1    12000
Name: count, dtype: int64

Validation class distribution:
label
0    1500
1    1500
Name: count, dtype: int64

Testing class distribution:
label
0    1500
1    1500
Name: count, dtype: int64


## 3. Import model

In [32]:
sys.path.append(os.path.abspath("../Dataset/conplag_version_2/"))

from scripts.codebert import create_model

tokenizer, model = create_model()

print(type(model))

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 13775.43it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


<class 'transformers.models.roberta.modeling_roberta.RobertaForSequenceClassification'>


## 4. Compile model

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

print(device)

cpu


In [34]:
print(model)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
  

## 5. Tokenizar

In [41]:
from torch.utils.data import Dataset, DataLoader

# Classes and methods to tokenize dataframes

class CodeAiDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.codes = dataframe["code"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.codes[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx]).long()
        }
    
class CodePlagiarismDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.code1 = dataframe["code1"].tolist()
        self.code2 = dataframe["code2"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.code1[idx],
            self.code2[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx]).long()
        }

In [42]:
ai_train_tokenized = CodeAiDataset(ai_train, tokenizer)
ai_val_tokenized = CodeAiDataset(ai_val, tokenizer)
ai_test_tokenized = CodeAiDataset(ai_test, tokenizer)

plagiarism_train_tokenized = CodePlagiarismDataset(plagiarism_train, tokenizer)
plagiarism_val_tokenized = CodePlagiarismDataset(plagiarism_val, tokenizer)
plagiarism_test_tokenized = CodePlagiarismDataset(plagiarism_test, tokenizer)

ai_train_loader = DataLoader(
    ai_train_tokenized,
    batch_size=8,
    shuffle=True
)

ai_val_loader = DataLoader(
    ai_val_tokenized,
    batch_size=8,
    shuffle=True
)

ai_test_loader = DataLoader(
    ai_test_tokenized,
    batch_size=8,
    shuffle=False
)

plagiarism_train_loader = DataLoader(
    plagiarism_train_tokenized,
    batch_size=8,
    shuffle=True
)

plagiarism_val_loader = DataLoader(
    plagiarism_val_tokenized,
    batch_size=8,
    shuffle=True
)

plagiarism_test_loader = DataLoader(
    plagiarism_test_tokenized,
    batch_size=8,
    shuffle=False
)

## 6. Training

In [ ]:
# Training con ai_train dataframe

epochs = 3

train_losses = []
train_accuracies = []

for epoch in range(epochs):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(ai_train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = total_loss / len(ai_train_loader)
    epoch_accuracy = correct / total

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    print(f"Epoch {epoch + 1}/{epochs}")
    print(f"Train Loss: {epoch_loss:.4f}")
    print(f"Train Accuracy: {epoch_accuracy:.4f}")

  0%|          | 14/3000 [08:01<28:29:52, 34.36s/it]


KeyboardInterrupt: 